In [1]:
!pip install pygame
!pip install rlcard

In [2]:
import sys
sys.path.append('..')

from environment import TrafficEnvironment
from keychain import Keychain as kc
import os
from services import Trainer
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.env_checker import check_env
from pettingzoo.test import parallel_api_test
from gymnasium.spaces import Box, Discrete
from pettingzoo.classic import leduc_holdem_v4
from torch import nn
import gymnasium as gym
import supersuit as ss
from stable_baselines3 import PPO
from Sumo_controller import Sumo
from stable_baselines3 import A2C
from utilities import confirm_env_variable
from utilities import get_params
from stable_baselines3 import SAC
import torch as th

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

kc.PARAMS_PATH = '../params.json'

confirm_env_variable(kc.SUMO_HOME, append="tools")
params = get_params(kc.PARAMS_PATH)

[CONFIRMED] Environment variable exists: SUMO_HOME
[SUCCESS] Added module directory: C:\Program Files (x86)\Eclipse\Sumo\tools


In [3]:
#Define the number of paths used in this experiment
simulator_params = params[kc.SIMULATION_PARAMETERS]

for key, value in simulator_params.items():
    if key == "sumo_config_path":
        simulator_params[key] = "../Network_and_config/csomor1.sumocfg"
    if key =='edge_file_path':
        simulator_params[key] = "../Network_and_config/csomor1.edg.xml"
    if key == 'connection_file_path':
        simulator_params[key] = "../Network_and_config/csomor1.con.xml"
    if key == 'route_file_path':
        simulator_params[key] = "../Network_and_config/csomor1.rou.xml"
    if key == 'routes_xml_save_path':
        simulator_params[key] = "../Network_and_config/route.rou.xml"
    
params[kc.SIMULATION_PARAMETERS] = simulator_params

In [4]:
Sumo_sim=Sumo(params)
Sumo_sim.Sumo_start()

In [5]:
number_of_paths = 3
origins = ["279952229#0", "115604053"]
destinations = ["-115602933#2", "-441496282#1"]
num_agents = 600

In [6]:
#Define the number of paths used in this experiment
simulator_params = params[kc.SIMULATION_PARAMETERS]

for key, value in simulator_params.items():
    if key == "number_of_paths":
        simulator_params[key] = number_of_paths
    if key =='origins':
        simulator_params[key] = origins
    if key == 'destinations':
        simulator_params[key] = destinations

params[kc.SIMULATION_PARAMETERS] = simulator_params

In [7]:
agent_generation_params = params[kc.AGENTS_GENERATION_PARAMETERS]

for key, value in agent_generation_params.items():
    if key == "num_agents":
        agent_generation_params[key] = num_agents

params[kc.AGENTS_GENERATION_PARAMETERS] = agent_generation_params

### Environment Creation

In [8]:
env = TrafficEnvironment(params[kc.ENVIRONMENT_PARAMETERS], params[kc.SIMULATION_PARAMETERS], params[kc.AGENTS_GENERATION_PARAMETERS])

[SUCCESS] Generated 4 routes
[SUCCESS] Generated & saved 12 paths to: training_records\paths.csv
[SUCCESS] Simulator is ready to simulate!
[SUCCESS] Environment initiated!
[INFO] Free-flow times:  {(0, 0): [2.080892356942777, 4.584748859947147, 1.9685474189675871], (0, 1): [3.3418967587034816, 0.7787715086034414, 0.7676270508203282], (1, 0): [0.7396357391048572, 0.6873687704023294, 0.9195730415996491], (1, 1): [2.052040701089651, 1.9711822958124936, 2.1767673087953683]}

[SUCCESS] Free flow times calculated!
[INFO] Free-flow times:  {(0, 0): [2.080892356942777, 4.584748859947147, 1.9685474189675871], (0, 1): [3.3418967587034816, 0.7787715086034414, 0.7676270508203282], (1, 0): [0.7396357391048572, 0.6873687704023294, 0.9195730415996491], (1, 1): [2.052040701089651, 1.9711822958124936, 2.1767673087953683]}
[SUCCESS] Generated agent data and saved to: training_records\agents_data.csv
[SUCCESS] Created agent objects (600)

Agent 0 has origin 0 and destination 1.



Agent 1 has origin 0 an

In [9]:
env = ss.pettingzoo_env_to_vec_env_v1(env)
env = ss.concat_vec_envs_v1(env, 1, num_cpus=0, base_class="stable_baselines3")

vec_env_fns:  [<function vec_env_args.<locals>.env_fn at 0x00000247C0813EC0>]
vnev is:  <ConcatVecEnv instance>
num_envs is:  600
action_space `is:  Discrete(3)


In [10]:
model = DQN("MlpPolicy", env, verbose=0)

The prints are from common/off_policy_algorithm.py -> function _sample_action()

For a specific number of episodes it takes random actions and then uses predit to learn.

Then it takes actions based on the base_class.py predict() function.

Predict takes place in the common/policies.py file.

In [11]:
model.learn(total_timesteps=500000)

n_envs is:  600
self.num_timesteps is:  0 learning_starts is:  50000
buffer action are:  [2 0 2 2 1 0 0 1 2 1 2 0 0 1 2 0 2 0 2 1 2 2 2 2 1 0 0 2 0 0 0 2 1 0 1 2 1
 2 1 2 2 1 2 0 1 2 1 1 1 0 0 2 2 2 0 1 2 2 0 2 2 1 1 2 2 2 1 1 0 2 0 0 1 2
 1 1 0 1 0 1 1 0 1 0 0 1 1 2 1 2 0 2 0 0 1 1 1 0 1 2 1 1 0 0 0 0 0 2 1 1 0
 0 1 0 0 1 2 1 1 0 0 0 0 2 0 1 1 1 2 2 2 0 2 0 0 1 1 2 1 1 1 1 1 0 0 0 0 2
 0 1 0 2 2 1 1 1 2 0 1 0 0 0 1 2 0 1 2 2 2 2 1 0 0 2 0 2 1 2 0 2 2 1 1 1 2
 1 2 2 0 1 0 1 2 0 0 2 2 1 0 1 2 0 1 2 2 0 1 0 1 1 0 0 0 2 1 0 0 1 2 2 0 0
 1 2 1 0 2 2 1 1 1 2 2 1 2 0 1 0 2 0 2 0 2 0 2 2 0 1 2 2 2 1 2 1 1 1 1 0 1
 1 2 2 0 0 2 0 0 2 1 2 0 1 2 0 0 0 2 0 1 0 1 1 2 1 2 0 1 0 1 1 0 1 0 2 1 2
 1 2 2 0 1 1 1 2 1 2 2 2 2 2 0 2 2 1 1 2 0 0 0 0 2 1 1 2 2 1 2 0 2 1 1 2 0
 1 1 1 0 2 1 1 0 1 2 1 0 0 0 1 2 2 1 1 0 2 0 0 1 0 2 1 1 0 0 0 2 2 2 2 0 1
 2 2 2 2 2 0 1 1 0 1 0 2 2 0 1 2 1 2 1 0 1 1 1 1 1 2 2 1 2 2 2 0 1 0 1 1 2
 2 2 2 1 1 1 2 2 0 1 1 2 1 0 1 0 0 2 2 1 0 2 2 0 0 1 0 1 0 2 1 2 1 1 0 1 2
 1 2 1 1 1 

KeyboardInterrupt: 

In [12]:
model = PPO("MlpPolicy", env, verbose=0)

env is:  <supersuit.vector.sb3_vector_wrapper.SB3VecEnvWrapper object at 0x0000017C40A9A650> 


action distribution is:  <stable_baselines3.common.distributions.CategoricalDistribution object at 0x0000017C4E27F710> 



last_layer_dim_pi:  3
3 




curr_layer_dim:  64
curr_layer_dim:  64



curr_layer_dim:  64
curr_layer_dim:  64
policy_net:  Sequential(
  (0): Linear(in_features=3, out_features=64, bias=True)
  (1): Tanh()
  (2): Linear(in_features=64, out_features=64, bias=True)
  (3): Tanh()
)
value_net:  Sequential(
  (0): Linear(in_features=3, out_features=64, bias=True)
  (1): Tanh()
  (2): Linear(in_features=64, out_features=64, bias=True)
  (3): Tanh()
)


In [11]:
model.learn(total_timesteps=500000)

inside forward



if tensor([[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]], dtype=torch.int32) FlattenExtractor(
  (flatten): Flatten(start_dim=1, end_dim=-1)
) 



features are:  tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        ...,
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]]) 



[1 0 0 0 2 0 0 2 0 2 0 1 1 0 2 2 1 0 2 1 1 1 2 0 1 1 1 1 0 2 1 2 1 1 2 0 0
 0 1 2 2 2 1 0 1 1 0 2 1 0 2 0 1 2 2 0 0 0 0 2 1 2 1 2 2 2 0 1 0 1 0 0 0 1
 0 0 1 2 1 1 2 1 1 1 1 2 1 1 1 1 0 2 2 1 0 2 1 0 1 1 0 2 1 1 1 1 1 1 1 1 1
 1 0 0 0 1 1 1 2 2 0 0 2 0 1 1 2 2 2 2 0 2 0 2 0 0 1 1 1 2 1 2 1 2 0 1 2 1
 1 1 1 0 2 0 0 0 0 1 2 2 0 0 2 0 0 2 0 0 0 1 2 1 2 1 0 2 0 1 1 0 0 2 1 2 2
 2 1 1 2 0 1 2 0 0 1 1 1 0 1 1 0 1 1 2 2 0 1 0 0 2 2 1 1 0 0 0 2 0 0 1 0 2
 0 1 0 2 0 1 1 0 2 0 0 0 1 1 0 0 0 2 0 1 1 0 2 1 0 2 2 2 0 0 1 2 0 0 2 1 1
 0 0 1 1 1 1 0 2 0 0 1 0 2 2 1 1 0 2 2 1 0 0 1 0 0 1 2 1 1 1 1 0 0 0 1 2

inside forward



if tensor([[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]], dtype=torch.int32) FlattenExtractor(
  (flatten): Flatten(start_dim=1, end_dim=-1)
) 



features are:  tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        ...,
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]]) 



[1 0 1 0 2 2 0 1 0 2 0 1 2 0 2 1 1 1 0 1 1 2 0 1 0 0 0 2 0 2 1 2 0 2 2 2 2
 0 1 2 0 2 1 0 0 2 1 2 0 0 1 2 1 1 2 0 1 0 1 2 1 2 2 1 0 1 0 1 2 2 0 0 2 2
 2 0 1 0 0 0 0 0 2 1 1 2 2 1 0 2 2 2 1 2 1 1 0 1 2 2 0 0 0 0 0 0 1 0 2 2 2
 2 0 0 2 0 0 0 1 2 1 2 0 1 1 2 0 0 2 2 0 2 1 0 2 1 1 2 0 2 0 2 0 0 2 2 0 1
 1 0 0 0 0 2 1 0 2 0 2 2 0 0 1 2 0 2 1 1 1 0 0 2 2 0 2 2 0 2 1 1 2 2 2 1 1
 1 2 2 1 0 1 1 0 1 1 1 1 0 0 1 2 0 1 2 0 2 1 0 1 0 2 2 1 2 0 1 2 0 1 2 2 1
 2 2 0 0 2 1 1 2 0 0 2 2 0 0 1 1 2 0 1 0 2 0 0 2 0 1 0 0 2 1 0 2 0 1 2 0 2
 2 1 1 1 1 2 0 1 1 2 2 0 0 0 0 1 0 0 1 1 1 2 1 0 1 2 2 2 2 1 0 2 2 1 1 0

KeyboardInterrupt: 